In [54]:
"""
# PhishGuard AI - Train and Compare Model

## Objective

This notebook trains and compares two machine learning classifiers:

1. Decision Tree Classifier
2. Random Forest Classifier

The models are evaluated using:

- Accuracy
- Precision
- Recall
- F1-score

The model with the highest phishing-class F1 score will be selected as the deployment model.

## Label Convention

- 0 = Legitimate
- 1 = Phishing
"""

'\n# PhishGuard AI - Train and Compare Model\n\n## Objective\n\nThis notebook trains and compares two machine learning classifiers:\n\n1. Decision Tree Classifier\n2. Random Forest Classifier\n\nThe models are evaluated using:\n\n- Accuracy\n- Precision\n- Recall\n- F1-score\n\nThe model with the highest phishing-class F1 score will be selected as the deployment model.\n\n## Label Convention\n\n- 0 = Legitimate\n- 1 = Phishing\n'

In [55]:

import json
import os
import string
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier



In [56]:

sys.path.insert(0, os.path.abspath("../backend"))
from app.features import URL_ONLY_FEATURES, compute_features  # noqa: E402
print(len(URL_ONLY_FEATURES))


23


In [57]:
RANDOM_STATE = 42
CSV = "../PhiUSIIL_Phishing_URL_Dataset.csv"
OUT = "../outputs"
os.makedirs(OUT, exist_ok=True)

In [58]:
if not os.path.exists(CSV):
    raise SystemExit(f"Dataset not found: {CSV} (place PhiUSIIL CSV in repo root)")

df = pd.read_csv(CSV, encoding="utf-8").drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=["URL"]).reset_index(drop=True)
extra_legitimate = pd.DataFrame({
    "URL": [
        "https://developer.mozilla.org/en-US/docs/Web/Security",
        "https://numpy.org/doc/",
        "https://scikit-learn.org/",
        "https://python.org",
        "https://pytorch.org/docs/",
        "https://tensorflow.org/api_docs",
        "https://flask.palletsprojects.com/",
        "https://github.com/login",
        "https://github.com/python/cpython",
        "https://stackoverflow.com/questions"
    ],
    "label": [
        1,1,1,1,1,1,1,1,1,1
    ]
})


df = pd.concat(
    [df, extra_legitimate],
    ignore_index=True
)
print(f"Loaded {len(df):,} rows after adding extra legitimate samples")
print(df.tail(10))
print(df["label"].value_counts())

Loaded 235,805 rows after adding extra legitimate samples
       FILENAME                                                URL  URLLength  \
235795      NaN  https://developer.mozilla.org/en-US/docs/Web/S...        NaN   
235796      NaN                             https://numpy.org/doc/        NaN   
235797      NaN                          https://scikit-learn.org/        NaN   
235798      NaN                                 https://python.org        NaN   
235799      NaN                          https://pytorch.org/docs/        NaN   
235800      NaN                    https://tensorflow.org/api_docs        NaN   
235801      NaN                 https://flask.palletsprojects.com/        NaN   
235802      NaN                           https://github.com/login        NaN   
235803      NaN                  https://github.com/python/cpython        NaN   
235804      NaN                https://stackoverflow.com/questions        NaN   

       Domain  DomainLength  IsDomainIP  TLD  URLS

In [59]:
y = 1 - df["label"].astype(int)
print(f"Class balance after flip -> Legit(0): {int((y==0).sum()):,} | "
      f"Phish(1): {int((y==1).sum()):,}")

Class balance after flip -> Legit(0): 134,860 | Phish(1): 100,945


In [60]:
print("\nBuilding TLD lookup...")
tld_lookup = (
    df.assign(_legit=(y == 0).astype(int))
      .groupby(df["URL"].apply(
          lambda u: u.split("//")[-1].split("/")[0].split(":")[0].rsplit(".", 1)[-1].lower()
          if "." in u.split("//")[-1].split("/")[0] else ""
      ))["_legit"]
      .mean()
      .to_dict()
)
print(f"  {len(tld_lookup)} distinct TLDs")

# Character -> P(char | legit URL). Built from legitimate URLs only, with
# +1 smoothing so any character seen at inference has a non-zero prob.
print("\nBuilding character-frequency lookup from legit URLs...")
counter = {c: 1 for c in string.printable}
total = sum(counter.values())
legit_urls = df.loc[y == 0, "URL"].astype(str)
for u in legit_urls.sample(min(50_000, len(legit_urls)), random_state=RANDOM_STATE):
    for c in u.lower():
        counter[c] = counter.get(c, 0) + 1
        total += 1
char_lookup = {c: v / total for c, v in counter.items()}
print(f"  {len(char_lookup)} distinct characters, total mass = "
      f"{sum(char_lookup.values()):.4f}")

# Save the lookups so features.py can load them at inference
joblib.dump(tld_lookup, os.path.join(OUT, "tld_probability_lookup.joblib"))
joblib.dump(char_lookup, os.path.join(OUT, "char_probability_lookup.joblib"))


Building TLD lookup...
  709 distinct TLDs

Building character-frequency lookup from legit URLs...
  100 distinct characters, total mass = 1.0000


['../outputs\\char_probability_lookup.joblib']

In [61]:
print(f"\nExtracting features from {len(df):,} URLs using compute_features()...")
feats = [compute_features(u, tld_lookup, char_lookup) for u in df["URL"].astype(str)]
X = pd.DataFrame(feats)[URL_ONLY_FEATURES].astype(float)
print(f"  Feature matrix shape: {X.shape}")
print("Feature number:", len(URL_ONLY_FEATURES))
print("X shape:", X.shape)


Extracting features from 235,805 URLs using compute_features()...
  Feature matrix shape: (235805, 23)
Feature number: 23
X shape: (235805, 23)


In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
print(f"\nTrain: {len(X_train):,} | Test: {len(X_test):,}")


def score(model, name):
    """Fit, evaluate on the held-out test set, return a metrics dict."""
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]  # P(phish)
    m = {
        "name": name,
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, pos_label=1)),
        "recall": float(recall_score(y_test, pred, pos_label=1)),
        "f1_score": float(f1_score(y_test, pred, pos_label=1)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_features": len(URL_ONLY_FEATURES),
    }
    print(f"\n{name}")
    print(classification_report(
        y_test, pred, target_names=["Legitimate(0)", "Phishing(1)"], digits=4,
    ))
    return model, m




Train: 188,644 | Test: 47,161


In [63]:
dt = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

dt_model, dt_metrics = score(dt, "DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)")
rf_model, rf_metrics = score(rf, "RandomForestClassifier(n_estimators=200, max_depth=20)")

# Persist both so the Admin Model Analytics page can compare them
joblib.dump(dt_model, os.path.join(OUT, "dt_model.joblib"))
joblib.dump(rf_model, os.path.join(OUT, "rf_model.joblib"))

# Winner (by F1 on the phishing class) becomes the deployment model
winner_name, winner_model = (
    ("RandomForest", rf_model)
    if rf_metrics["f1_score"] >= dt_metrics["f1_score"]
    else ("DecisionTree", dt_model)
)
print(f"\n>>> Winner (higher phishing-class F1): {winner_name}")

joblib.dump(winner_model, os.path.join(OUT, "deployment_model.joblib"))
joblib.dump(URL_ONLY_FEATURES, os.path.join(OUT, "deployment_features.joblib"))



DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)
               precision    recall  f1-score   support

Legitimate(0)     0.9913    0.9939    0.9926     26972
  Phishing(1)     0.9918    0.9883    0.9901     20189

     accuracy                         0.9915     47161
    macro avg     0.9916    0.9911    0.9913     47161
 weighted avg     0.9915    0.9915    0.9915     47161


RandomForestClassifier(n_estimators=200, max_depth=20)
               precision    recall  f1-score   support

Legitimate(0)     0.9938    0.9968    0.9953     26972
  Phishing(1)     0.9957    0.9917    0.9937     20189

     accuracy                         0.9946     47161
    macro avg     0.9947    0.9942    0.9945     47161
 weighted avg     0.9946    0.9946    0.9946     47161


>>> Winner (higher phishing-class F1): RandomForest


['../outputs\\deployment_features.joblib']

In [64]:
metrics_blob = {
    "winner": winner_name,
    "label_convention": {"0": "Legitimate", "1": "Phishing"},
    "positive_class": "Phishing",
    "decision_tree": dt_metrics,
    "random_forest": rf_metrics,
}
joblib.dump(metrics_blob, os.path.join(OUT, "deployment_metrics.joblib"))
with open(os.path.join(OUT, "evaluation_metrics.txt"), "w") as fh:
    fh.write(json.dumps(metrics_blob, indent=2))

bg = X_train.sample(1000, random_state=RANDOM_STATE).to_numpy()
np.save(os.path.join(OUT, "lime_background.npy"), bg)

In [66]:
print("\nSanity check on a few well-known URLs (0 = Legit, 1 = Phish):")

check_urls = [
    "https://chatgpt.com",
    "https://www.google.com/search?q=google+translate",
    "https://github.com/anthropics/anthropic-sdk-python",
    "https://en.wikipedia.org/wiki/Main_Page",
    "http://paypa1-secure-login.win/verify.php?id=aGVsbG8=&acct=443",
    "http://192.168.0.1/login.php?redirect=%2Fadmin",
]


for u in check_urls:

    fv = np.array([
        [
            compute_features(u, tld_lookup, char_lookup)[n]
            for n in URL_ONLY_FEATURES
        ]
    ])

    prob = winner_model.predict_proba(fv)[0][1]

    threshold = 0.8

    p = 1 if prob >= threshold else 0

    print(
        f"  P(phish)={prob:.3f}  pred={int(p)}  {u}"
    )


print("\nSaved to ./outputs:")

for f in [
    "deployment_model.joblib",
    "deployment_features.joblib",
    "deployment_metrics.joblib",
    "dt_model.joblib",
    "rf_model.joblib",
    "tld_probability_lookup.joblib",
    "char_probability_lookup.joblib",
    "lime_background.npy",
    "evaluation_metrics.txt",
]:

    print(f"  - {f}")



test_urls = [
    "https://developer.mozilla.org/en-US/docs/Web/Security",
    "https://numpy.org/doc/",
    "https://scikit-learn.org/",
    "https://python.org",
    "https://github.com/login"
]


print("\nFalse Positive Test URLs:")

threshold = 0.8


for url in test_urls:

    fv = np.array([
        [
            compute_features(url, tld_lookup, char_lookup)[f]
            for f in URL_ONLY_FEATURES
        ]
    ])


    rf_prob = rf_model.predict_proba(fv)[0][1]


    rf_result = 1 if rf_prob >= threshold else 0


    print("\nURL:", url)

    print(
        f"Random Forest P(phish)={rf_prob:.3f}"
    )

    print(
        "Random Forest:",
        "Phishing" if rf_result == 1 else "Legitimate"
    )


    features = compute_features(
        url,
        tld_lookup,
        char_lookup
    )


    print(
        "DocumentationIndicator:",
        features["DocumentationIndicator"]
    )

    print(
        "SuspiciousKeywordCount:",
        features["SuspiciousKeywordCount"]
    )

    print(
        "BrandDomainMismatch:",
        features["BrandDomainMismatch"]
    )



importance = pd.Series(
    rf_model.feature_importances_,
    index=URL_ONLY_FEATURES
)


print("\nRandom Forest Feature Importance:")

print(
    importance.sort_values(
        ascending=False
    )
)


Sanity check on a few well-known URLs (0 = Legit, 1 = Phish):
  P(phish)=0.945  pred=1  https://chatgpt.com
  P(phish)=0.999  pred=1  https://www.google.com/search?q=google+translate
  P(phish)=0.993  pred=1  https://github.com/anthropics/anthropic-sdk-python
  P(phish)=0.781  pred=0  https://en.wikipedia.org/wiki/Main_Page
  P(phish)=1.000  pred=1  http://paypa1-secure-login.win/verify.php?id=aGVsbG8=&acct=443


c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\De

  P(phish)=1.000  pred=1  http://192.168.0.1/login.php?redirect=%2Fadmin

Saved to ./outputs:
  - deployment_model.joblib
  - deployment_features.joblib
  - deployment_metrics.joblib
  - dt_model.joblib
  - rf_model.joblib
  - tld_probability_lookup.joblib
  - char_probability_lookup.joblib
  - lime_background.npy
  - evaluation_metrics.txt

False Positive Test URLs:

URL: https://developer.mozilla.org/en-US/docs/Web/Security
Random Forest P(phish)=0.703
Random Forest: Legitimate
DocumentationIndicator: 1
SuspiciousKeywordCount: 1
BrandDomainMismatch: 0

URL: https://numpy.org/doc/
Random Forest P(phish)=0.755
Random Forest: Legitimate
DocumentationIndicator: 0
SuspiciousKeywordCount: 0
BrandDomainMismatch: 0

URL: https://scikit-learn.org/
Random Forest P(phish)=0.662
Random Forest: Legitimate
DocumentationIndicator: 1
SuspiciousKeywordCount: 0
BrandDomainMismatch: 0

URL: https://python.org
Random Forest P(phish)=0.719
Random Forest: Legitimate
DocumentationIndicator: 0
SuspiciousKey

c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\De